## Week 2 Technical Q&A Prototype

This notebook applies all Week 2 learnings to build a full-featured technical Q&A assistant:

| Feature | Week 2 Day |
|---|---|
| Multi-model API clients (OpenAI, OpenRouter, Ollama) | Day 1 |
| Streaming responses with generators | Day 2 |
| Gradio `ChatInterface` with conversation history | Day 3 |
| Tool / function calling | Day 4 |

---

In [ ]:
import os
import json
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

openai_client = OpenAI()

openrouter_client = OpenAI(
    api_key=os.getenv('OPENROUTER_API_KEY'),
    base_url="https://openrouter.ai/api/v1",
)

ollama_client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1",
)

# Model registry: label → (client, model-id)
MODELS = {
    "GPT-4o-mini": (openai_client, "gpt-4o-mini"),
    "GPT-4.1-mini (OpenRouter)": (openrouter_client, "openai/gpt-4.1-mini"),
    "Llama3.2 (Local)": (ollama_client, "llama3.2"),
}

# ---------------------------------------------------------------------------
# System prompt — expertise injected via system message
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = """You are an expert technical explainer and tutor. \
Break down complex concepts into clear explanations with concrete examples. \
When code is relevant, include concise snippets. \
You have access to a glossary tool for quick term look-ups."""

In [ ]:
TECH_GLOSSARY = {
    "api": "Application Programming Interface — a contract that lets programs talk to each other.",
    "llm": "Large Language Model — a neural network trained on massive text to understand and generate language.",
    "transformer": "A neural-network architecture based on self-attention, foundational to modern LLMs.",
    "fine-tuning": "Further training a pre-trained model on task-specific data to specialise its behaviour.",
    "embeddings": "Dense numerical vectors that encode the semantic meaning of text.",
    "rag": "Retrieval-Augmented Generation — coupling an LLM with an external knowledge retriever.",
    "prompt engineering": "Crafting inputs that reliably guide an LLM toward desired outputs.",
    "token": "The atomic unit an LLM processes; roughly a word or sub-word piece.",
    "inference": "Running a trained model to produce outputs from new inputs.",
    "gradient descent": "An iterative optimisation algorithm that reduces loss by following the negative gradient.",
    "attention": "A mechanism that lets each token weigh the relevance of every other token in the sequence.",
    "hallucination": "When an LLM generates plausible-sounding but factually incorrect information.",
}

def lookup_definition(term: str) -> str:
    result = TECH_GLOSSARY.get(term.lower().strip())
    if result:
        return f"**{term.title()}**: {result}"
    return f"'{term}' not found in the glossary — ask me for a detailed explanation instead."

tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_definition",
            "description": "Quick-look up a technical term definition from the built-in glossary.",
            "parameters": {
                "type": "object",
                "properties": {
                    "term": {
                        "type": "string",
                        "description": "The exact technical term, e.g. 'transformer', 'RAG', 'token'.",
                    }
                },
                "required": ["term"],
            },
        },
    }
]

def handle_tool_call(tool_call):
    args = json.loads(tool_call.function.arguments)
    content = lookup_definition(args["term"])
    return {"role": "tool", "content": content, "tool_call_id": tool_call.id}

In [ ]:
def chat(message, history, model_choice, use_tools):
    """Yield streamed response chunks. Supports tool calling when enabled."""
    client, model = MODELS[model_choice]

    # Build messages: system + history + current user message
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in history:
        messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": message})

    # Tool calling is only available for cloud models (not Ollama)
    active_tools = tools if (use_tools and "Local" not in model_choice) else None

    if active_tools:
        # Non-streaming path so we can handle tool calls
        response = client.chat.completions.create(
            model=model, messages=messages, tools=active_tools
        )
        while response.choices[0].finish_reason == "tool_calls":
            tool_msg = response.choices[0].message
            messages.append(tool_msg)
            for tc in tool_msg.tool_calls:
                messages.append(handle_tool_call(tc))
            response = client.chat.completions.create(
                model=model, messages=messages, tools=active_tools
            )
        yield response.choices[0].message.content
    else:
        # Streaming path
        stream = client.chat.completions.create(
            model=model, messages=messages, stream=True
        )
        result = ""
        for chunk in stream:
            result += chunk.choices[0].delta.content or ""
            yield result

In [ ]:
model_dropdown = gr.Dropdown(
    choices=list(MODELS.keys()),
    value="GPT-4o-mini",
    label="Model",
)

tools_toggle = gr.Checkbox(
    value=True,
    label="Enable Tool Calling (Day 4 bonus — glossary look-up)",
)

gr.ChatInterface(
    fn=chat,
    type="messages",
    title="🤖 Technical Q&A Assistant — Week 2",
    description=(
        "Ask any technical question. The assistant streams its answer, "
        "remembers the conversation, and can call a glossary tool for quick definitions. "
        "Switch between models using the dropdown below."
    ),
    additional_inputs=[model_dropdown, tools_toggle],
    examples=[
        ["Explain how the transformer architecture works"],
        ["What is the difference between fine-tuning and RAG?"],
        ["How does gradient descent minimise loss?"],
        ["Define the term 'token' in the context of LLMs"],
        ["What is prompt engineering and why does it matter?"],
    ],
    theme=gr.themes.Soft(),
).launch()